In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
import os
import uuid

from utils import (
    get_hdfs_base_uri,
    create_namespace,
    read_csv_raw,
    normalize_dataframe_columns,
    cast_all_columns_to_string,
    add_bronze_metadata,
    add_record_hash,
    filter_new_source_files,
    append_iceberg,
    default_spark_local_dir,
    is_truthy,
)

SPARK_LOCAL_DIR = default_spark_local_dir()
os.environ["SPARK_LOCAL_DIRS"] = SPARK_LOCAL_DIR
os.makedirs(SPARK_LOCAL_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .config("spark.local.dir", SPARK_LOCAL_DIR)
    .getOrCreate()
)

HDFS_BASE_URI = get_hdfs_base_uri()

CATALOG = "spark_catalog"
BRONZE_DB = "bronze"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_DB}.beneficiarios"

RAW_PATH = f"{HDFS_BASE_URI}/dados/raw/ans/"
BRONZE_LOCATION = f"{HDFS_BASE_URI}/user/hive/warehouse/{BRONZE_DB}.db"

SOURCE_SYSTEM = os.getenv("SOURCE_SYSTEM", "ANS")

BATCH_ID = os.getenv("BATCH_ID")

if not BATCH_ID:
    BATCH_ID = datetime.now().strftime("%Y%m%d%H%M%S") + "_" + uuid.uuid4().hex[:8]

FORCE_REPROCESS_FILES = is_truthy(os.getenv("FORCE_REPROCESS_FILES"))

create_namespace(
    spark=spark,
    catalog=CATALOG,
    database=BRONZE_DB,
    location=BRONZE_LOCATION,
)


df_raw = read_csv_raw(
    spark=spark,
    path=RAW_PATH,
    sep=";",
)

df_normalized = normalize_dataframe_columns(df_raw)
df_bronze = cast_all_columns_to_string(df_normalized)

raw_payload_columns = df_bronze.columns

df_bronze = (
    df_bronze
    .transform(lambda df: add_bronze_metadata(
        df=df,
        source_system=SOURCE_SYSTEM,
        batch_id=BATCH_ID,
    ))
    .transform(lambda df: add_record_hash(
        df=df,
        payload_columns=raw_payload_columns,
        hash_column_name="_record_hash",
    ))
)

df_bronze_incremental = filter_new_source_files(
    spark=spark,
    source_df=df_bronze,
    target_table=BRONZE_TABLE,
    source_path_column="_source_path",
    force_reprocess=FORCE_REPROCESS_FILES,
)

rows_to_write = df_bronze_incremental.limit(1).count()

if rows_to_write == 0:
    print("Nenhum arquivo novo encontrado para carregar na Bronze.")
    print(f"Tabela Bronze mantida sem alteração: {BRONZE_TABLE}")
else:
    append_iceberg(
        spark=spark,
        source_df=df_bronze_incremental,
        target_table=BRONZE_TABLE,
    )

    file_count = df_bronze_incremental.select("_source_path").distinct().count()
    row_count = df_bronze_incremental.count()

    print(f"Batch carregado na Bronze: {BATCH_ID}")
    print(f"Arquivos processados: {file_count}")
    print(f"Linhas gravadas: {row_count}")
    print(f"Tabela: {BRONZE_TABLE}")

spark.sql(f"""
SELECT
    _batch_id,
    COUNT(*) AS linhas,
    COUNT(DISTINCT _source_path) AS arquivos
FROM {BRONZE_TABLE}
GROUP BY _batch_id
ORDER BY _batch_id DESC
LIMIT 10
""").show(truncate=False)
